In [1]:
!pip install pandas matplotlib seaborn

In [2]:
!pip install matplotlib-inline

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, dayofmonth
spark = SparkSession.builder.appName("ETL_SUPERSTORE") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


# Ruta base en Cloud Storage
bucket_path = "gs://bucket_01_practifinal/raw/"

# Leer el dataset
dt_superstore = spark.read.option("header", True).option("inferSchema", True).csv(bucket_path + "Sample_Superstore.csv")


25/12/16 06:10:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
dt_superstore.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [5]:
dt_superstore.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [6]:
# ETAPA 2 – TRANSFORMACIÓN EN MODELO ESTRELLA

# 1. Iniciar sesión Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ETL_SUPERSTORE").getOrCreate()

In [7]:
# 2. Leer y limpiar los CSV desde Cloud Storage
# 2.1 Definir función para limpiar columnas (quita BOM, espacios, etc.)
from pyspark.sql.functions import col

def limpiar_columnas(df):
    return df.select([col(c).alias(c.strip().replace("ï»¿", "").replace("\ufeff", "")) for c in df.columns])

In [8]:
#2.2 Leer datasets
bucket = "bucket_01_practifinal"
path_raw = f"gs://{bucket}/raw/"

# Superstore
superstore = spark.read.option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .csv(path_raw + "Sample_Superstore.csv")

superstore_clean = limpiar_columnas(superstore)



In [9]:
# 3. Crear tablas del modelo estrella
# 3.1 dim_tiempo
from pyspark.sql.functions import col, to_date, year, month, dayofmonth
superstore_clean = superstore_clean.withColumn("Order Date", to_date(col("Order Date"), "MM/dd/yyyy"))

# Crear dim_tiempo: Tabla de Dimensión de Tiempo
dim_tiempo = superstore_clean.select("Order Date").distinct() \
    .withColumn("ANO", year("Order Date")) \
    .withColumn("MES", month("Order Date")) \
    .withColumn("DIA", dayofmonth("Order Date")) \
    .withColumn("ID_TIEMPO", col("Order Date").cast("string"))


In [10]:
# Mostrar dim_tiempo
dim_tiempo.show()

+----------+----+---+---+----------+
|Order Date| ANO|MES|DIA| ID_TIEMPO|
+----------+----+---+---+----------+
|2017-09-11|2017|  9| 11|2017-09-11|
|2016-03-01|2016|  3|  1|2016-03-01|
|2014-09-26|2014|  9| 26|2014-09-26|
|2014-11-12|2014| 11| 12|2014-11-12|
|2017-08-11|2017|  8| 11|2017-08-11|
|2015-03-09|2015|  3|  9|2015-03-09|
|2016-04-25|2016|  4| 25|2016-04-25|
|2015-03-06|2015|  3|  6|2015-03-06|
|2017-01-06|2017|  1|  6|2017-01-06|
|2016-08-15|2016|  8| 15|2016-08-15|
|2015-04-09|2015|  4|  9|2015-04-09|
|2016-10-03|2016| 10|  3|2016-10-03|
|2014-08-01|2014|  8|  1|2014-08-01|
|2015-12-22|2015| 12| 22|2015-12-22|
|2016-05-03|2016|  5|  3|2016-05-03|
|2016-08-31|2016|  8| 31|2016-08-31|
|2017-09-28|2017|  9| 28|2017-09-28|
|2017-02-26|2017|  2| 26|2017-02-26|
|2014-06-03|2014|  6|  3|2014-06-03|
|2017-01-27|2017|  1| 27|2017-01-27|
+----------+----+---+---+----------+
only showing top 20 rows



In [11]:
# 1. **dim_producto**: Tabla de Dimensión de Producto
dim_producto = superstore_clean.select("Product ID", "Category", "Sub-Category", "Product Name").distinct()

In [12]:
dim_producto.show()

+---------------+---------------+------------+--------------------+
|     Product ID|       Category|Sub-Category|        Product Name|
+---------------+---------------+------------+--------------------+
|TEC-AC-10001767|     Technology| Accessories|SanDisk Ultra 64 ...|
|TEC-PH-10001552|     Technology|      Phones|I Need's 3d Hello...|
|OFF-PA-10001937|Office Supplies|       Paper|            Xerox 21|
|OFF-PA-10003039|Office Supplies|       Paper|          Xerox 1960|
|OFF-EN-10000483|Office Supplies|   Envelopes|White Envelopes, ...|
|OFF-AR-10001953|Office Supplies|         Art|Boston 1645 Delux...|
|FUR-BO-10004360|      Furniture|   Bookcases|Rush Hierlooms Co...|
|OFF-LA-10003388|Office Supplies|      Labels|             Avery 5|
|OFF-BI-10003460|Office Supplies|     Binders|   Acco 3-Hole Punch|
|OFF-PA-10000788|Office Supplies|       Paper|           Xerox 210|
|TEC-PH-10000148|     Technology|      Phones|Cyber Acoustics A...|
|TEC-AC-10001314|     Technology| Accessories|Ca

In [13]:
# 2. **dim_cliente_segmento**: Tabla de Dimensión de Cliente/Segmento
dim_cliente_segmento = superstore_clean.select("Customer ID", "Customer Name", "Segment").distinct()


In [14]:
dim_cliente_segmento.show()

+-----------+------------------+-----------+
|Customer ID|     Customer Name|    Segment|
+-----------+------------------+-----------+
|   PN-18775|    Parhena Norris|Home Office|
|   DP-13105|      Dave Poirier|  Corporate|
|   GH-14665|       Greg Hansen|   Consumer|
|   LS-17200|      Luke Schmidt|  Corporate|
|   GZ-14470|     Gary Zandusky|   Consumer|
|   DR-12880|   Dan Reichenbach|  Corporate|
|   GT-14635|    Grant Thornton|  Corporate|
|   MG-17875|     Michael Grace|Home Office|
|   SU-20665|Stephanie Ulpright|Home Office|
|   JM-15580|     Jill Matthias|   Consumer|
|   DJ-13420|         Denny Joy|  Corporate|
|   TS-21085|     Thais Sissman|   Consumer|
|   SC-20575|      Sonia Cooley|   Consumer|
|   NC-18415|       Nathan Cano|   Consumer|
|   BK-11260|    Berenike Kampe|   Consumer|
|   CS-12490|  Cindy Schnelling|  Corporate|
|   AG-10390|    Allen Goldenen|   Consumer|
|   NP-18700|        Nora Preis|   Consumer|
|   HF-14995|   Herbert Flentye|   Consumer|
|   BS-118

In [15]:
# 3. **dim_region**: Tabla de Dimensión de Región
dim_region = superstore_clean.select("Region", "Country", "City").distinct()


In [16]:
dim_region.show()

+-------+-------------+------------+
| Region|      Country|        City|
+-------+-------------+------------+
|   East|United States|      Medina|
|   West|United States|  Marysville|
|Central|United States|     Bedford|
|Central|United States|       Eagan|
|   East|United States|       Akron|
|  South|United States|    Gastonia|
|   West|United States| Santa Maria|
|   East|United States|Mount Vernon|
|Central|United States|      Canton|
|   West|United States|   Vancouver|
|   East|United States|   Rockville|
|   West|United States|    Temecula|
|Central|United States|  Cedar Hill|
|   East|United States|   Cambridge|
|  South|United States|Delray Beach|
|Central|United States|      Normal|
|Central|United States|     Decatur|
|Central|United States|   Lakeville|
|   East|United States|     Clinton|
|  South|United States| Summerville|
+-------+-------------+------------+
only showing top 20 rows



In [17]:
# Crear la tabla de hechos: fact_ventas
from pyspark.sql.functions import col, regexp_replace
fact_ventas = superstore_clean.select(
    "Order ID", 
    "Product ID", 
    "Customer ID", 
    "Region", 
    "Sales", 
    "Profit", 
    "Quantity", 
    "Order Date"
).distinct()

# Limpiar la columna 'Sales' eliminando cualquier carácter no numérico
fact_ventas = fact_ventas.withColumn(
    "Sales", 
    regexp_replace(col("Sales"), "[^0-9.]", "").cast("float")
)


In [18]:
fact_ventas.show()

+--------------+---------------+-----------+-------+------+--------+--------+----------+
|      Order ID|     Product ID|Customer ID| Region| Sales|  Profit|Quantity|Order Date|
+--------------+---------------+-----------+-------+------+--------+--------+----------+
|CA-2017-153787|OFF-AP-10001563|   AT-10735|   West| 97.16| 28.1764|       2|2017-05-19|
|US-2014-117135|OFF-ST-10002444|   NP-18325|  South| 36.84| 10.3152|       3|2014-06-21|
|CA-2017-161984|OFF-FA-10000624|   SJ-20125|   East|  7.16|    3.58|       2|2017-04-10|
|CA-2017-144638|FUR-FU-10003535|   MH-18115|   East|43.872| 11.5164|       2|2017-03-10|
|CA-2016-124485|OFF-PA-10004888|   NC-18340|   East|  6.48|  3.1104|       1|2016-11-24|
|CA-2015-124800|OFF-BI-10003984|   RW-19540|   West|77.031|-59.0571|       9|2015-09-26|
|CA-2016-106530|OFF-ST-10000649|   CL-12565|   East| 25.12|    1.57|       2|2016-05-08|
|CA-2017-152807|FUR-FU-10004415|   MC-18100|   East| 7.168|  0.9856|       2|2017-10-30|
|CA-2016-167507|OFF-B

In [19]:
# Guardar las tablas como archivos Parquet en Google Cloud Storage, dentro de la carpeta 'docs'
dim_tiempo.write.parquet("gs://bucket_01_practifinal/docs/dim_tiempo")
dim_producto.write.parquet("gs://bucket_01_practifinal/docs/dim_producto")
dim_cliente_segmento.write.parquet("gs://bucket_01_practifinal/docs/dim_cliente_segmento")
dim_region.write.parquet("gs://bucket_01_practifinal/docs/dim_region")
fact_ventas.write.parquet("gs://bucket_01_practifinal/docs/fact_ventas")

AnalysisException: path gs://bucket_01_practifinal/docs/dim_tiempo already exists.

In [20]:
import matplotlib
# Establecer un backend compatible
matplotlib.use('Agg')

# Ahora podemos crear el gráfico sin problemas
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import sum

# Agregar ventas por región
ventas_por_region = fact_ventas.groupBy("Region").agg(sum("Sales").alias("ventas_totales"))

# Convertir el DataFrame de PySpark a Pandas para graficar
ventas_por_region_pandas = ventas_por_region.toPandas()

# Crear el gráfico de barras con Seaborn
plt.figure(figsize=(10, 6))
sns.barplot(x="Region", y="ventas_totales", data=ventas_por_region_pandas)

# Títulos y etiquetas
plt.title('Ventas Totales por Región', fontsize=16)
plt.xlabel('Región', fontsize=12)
plt.ylabel('Ventas Totales (USD)', fontsize=12)

# Mostrar el gráfico
plt.xticks(rotation=45)
plt.show()
